# Maps and landmarks: notebook

The room of the Note: 7 m x 4 m with a door, a four-sided table, an L-shaped cabinet and two poles. The geometry lives in `images/roommap.py`; here we check every number of the Note.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import roommap as m

## 1. Feature-based and grid maps (Note, Section 3)

In [2]:
feature_map = [(x, y, s) for s, (x, y) in m.POLES.items()]
print('feature map:', feature_map, '->', 3 * len(feature_map), 'numbers')
B = m.bitmap(0.1, sub=4)
print('grid map at 0.1 m:', B.shape[1], 'x', B.shape[0], '=', B.size, 'cells')

feature map: [(1.0, 3.0, 'pink'), (6.0, 0.8, 'green')] -> 6 numbers


grid map at 0.1 m: 70 x 40 = 2800 cells


## 2. The table as four half-planes (Note, Section 4)

In [3]:
# edge from (5, 1) to (5.5, 2.5): direction (0.5, 1.5), outward normal (1.5, -0.5)
a, b = 1.5, -0.5
c = -(a * 5 + b * 1)
print('f2 before scaling:', a, b, c, ' x2 ->', 2 * a, 2 * b, 2 * c)
print('f2 at the other corner (5.5, 2.5):', 3 * 5.5 - 2.5 - 14, ' at (6, 1):', 3 * 6 - 1 - 14)
for p in [(4, 2), (5.5, 1.5), (2, 1)]:
    print(p, 'f =', m.f_table(*p), ' inside table:', m.in_table(*p))

f2 before scaling: 1.5 -0.5 -7.0  x2 -> 3.0 -1.0 -14.0
f2 at the other corner (5.5, 2.5): 0.0  at (6, 1): 3
(4, 2) f = [-1, -4, -7, -3]  inside table: True
(5.5, 1.5) f = [-0.5, 1.0, -8.0, -9.5]  inside table: False
(2, 1) f = [0, -9, -19, 4]  inside table: False


In [4]:
for p in [(6.2, 3.0), (6.7, 2.6), (5.8, 3.6)]:
    print(p, 'in R1:', m.in_rect(*p, m.CAB[0]), ' in R2:', m.in_rect(*p, m.CAB[1]), ' in cabinet:', m.in_cabinet(*p))

(6.2, 3.0) in R1: False  in R2: False  in cabinet: False
(6.7, 2.6) in R1: False  in R2: True  in cabinet: True
(5.8, 3.6) in R1: True  in R2: False  in cabinet: True


## 3. Bitmaps at two resolutions (Note, Section 5.2)

In [5]:
T = m.TABLE
table_area = 0.5 * abs(sum(T[i][0] * T[(i + 1) % 4][1] - T[(i + 1) % 4][0] * T[i][1] for i in range(4)))
cab_area = 1.4 * 0.6 + 0.6 * 1.6 - 0.6 * 0.6
true_area = table_area + cab_area + 2 * np.pi * m.POLE_R ** 2
print(f'true obstacle area: table {table_area} + cabinet {cab_area:.2f} + poles = {true_area:.2f} m2')
for cell, sub in [(0.5, 10), (0.1, 4)]:
    B = m.bitmap(cell, sub)
    print(f'cell {cell} m: {B.size} cells, {B.sum()} black, black area {B.sum() * cell * cell:.2f} m2')

true obstacle area: table 3.625 + cabinet 1.44 + poles = 5.13 m2
cell 0.5 m: 112 cells, 47 black, black area 11.75 m2


cell 0.1 m: 2800 cells, 617 black, black area 6.17 m2


## 4. Extracting the poles from a laser scan (Note, Section 6)

In [6]:
b, r = m.scan(m.ROBOT, noise=0.01, rng=np.random.default_rng(0))
x, y, th = m.ROBOT
for rng_, bear, n in m.extract_poles(b, r):
    lx, ly = x + rng_ * np.cos(th + bear), y + rng_ * np.sin(th + bear)
    print(f'pole: {n} beams, range {rng_:.3f} m, bearing {np.degrees(bear):.1f} deg -> map point ({lx:.2f}, {ly:.2f})')
for s, (px, py) in m.POLES.items():
    rr = np.hypot(px - x, py - y)
    direction = np.degrees(np.arctan2(py - y, px - x))
    print(f'{s}: true range {rr:.3f} m, direction {direction:.1f} deg, bearing {direction - 30:.1f} deg')

pole: 3 beams, range 4.004 m, bearing -33.0 deg -> map point (6.00, 0.79)
pole: 5 beams, range 2.232 m, bearing 87.0 deg -> map point (0.99, 2.99)
pink: true range 2.236 m, direction 116.6 deg, bearing 86.6 deg
green: true range 4.005 m, direction -2.9 deg, bearing -32.9 deg
